# Local RAG Chatbot — Ollama + Gemma 4 (E2B) + ChromaDB

## Overview

This notebook builds a **fully local, retrieval-augmented chatbot** that runs on the same machine — instead of specializing model *weights*, it specializes model *knowledge* at query time by retrieving relevant chunks from your own documents before generating an answer.

**Stack:**
- **Generation model:** `gemma4:e2b` served locally via [Ollama](https://ollama.com)
- **Embedding model:** `nomic-embed-text` (also served via Ollama)
- **Vector store:** ChromaDB (persistent, local DB engine)
- **No LangChain** — retrieval, chunking, and prompting are implemented directly so every step is transparent and easy to modify (same philosophy as your password-reset agent: prefer deterministic, inspectable logic over a black-box framework where possible).

## Prerequisites (run once, outside this notebook)

```bash
# 1. Install Ollama if you haven't: https://ollama.com/download
# 2. Start the Ollama server (in a terminal, keep it running)
ollama serve

# 3. Pull the models (the notebook will also do this automatically if missing)
ollama pull gemma4:e2b
ollama pull nomic-embed-text
```

## Workflow
1. Load documents from a local folder (PDF / Markdown / TXT)
2. Chunk the text
3. Embed each chunk and store it in ChromaDB
4. On each query: embed the question → retrieve top-k chunks → build a grounded prompt → generate with Gemma 4 E2B
5. Chat interactively, right inside the notebook

In [2]:
%pip install ollama chromadb pypdf tqdm

Defaulting to user installation because normal site-packages is not writeable
     --------------------------------------- 23.5/23.5 MB 12.8 MB/s eta 0:00:00
     ------------------------------------- 382.9/382.9 kB 12.0 MB/s eta 0:00:00
     ---------------------------------------- 44.5/44.5 kB 2.1 MB/s eta 0:00:00
     --------------------------------------- 13.8/13.8 MB 14.5 MB/s eta 0:00:00
  Using cached pypika-0.51.1-py2.py3-none-any.whl (60 kB)
  Using cached overrides-7.7.0-py3-none-any.whl (17 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl (37 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl (150 kB)
     ---------------------------------------- 4.6/4.6 MB 15.6 MB/s eta 0:00:00
  Using cached mmh3-5.2.1-cp311-cp311-win_amd64.whl (41 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl (10 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
  Using cached durationpy-0.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-prometheus 0.60b1 requires opentelemetry-sdk~=1.39.1, but you have opentelemetry-sdk 1.44.0 which is incompatible.
opentelemetry-instrumentation 0.60b1 requires opentelemetry-semantic-conventions==0.60b1, but you have opentelemetry-semantic-conventions 0.65b0 which is incompatible.

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import glob
import ollama
import chromadb
from pypdf import PdfReader
from tqdm import tqdm

## 1. Configuration

Drop your own PDFs / `.md` / `.txt` files into `data/knowledge_base/` before running the indexing cells below. If the folder is empty the first time you run this, the notebook will populate it with a couple of sample documents so you can test the pipeline end-to-end immediately.

In [4]:
GENERATION_MODEL = "gemma4:e2b"
EMBEDDING_MODEL = "nomic-embed-text"

KNOWLEDGE_BASE_DIR = "data/knowledge_base"
CHROMA_PERSIST_DIR = "data/chroma_db"
COLLECTION_NAME = "personal_knowledge_base"

CHUNK_SIZE = 800      # characters per chunk
CHUNK_OVERLAP = 120   # overlap between consecutive chunks
TOP_K = 4             # number of chunks retrieved per query

## 2. Make sure the required models are pulled

This checks `ollama list` and pulls anything missing, so the notebook is runnable on a fresh machine without a separate terminal step.

In [5]:
def ensure_model(model_name):
    local_models = [m["model"] for m in ollama.list()["models"]]
    if model_name not in local_models:
        print(f"Pulling {model_name} ... (first time only, may take a while)")
        for progress in ollama.pull(model_name, stream=True):
            status = progress.get("status", "")
            print(f"\r{status}", end="")
        print()
    else:
        print(f"{model_name} already available.")

ensure_model(GENERATION_MODEL)
ensure_model(EMBEDDING_MODEL)

Pulling gemma4:e2b ... (first time only, may take a while)
success manifest digest
Pulling nomic-embed-text ... (first time only, may take a while)
success manifest digest


## 3. Knowledge base

If `data/knowledge_base/` is empty, we seed it with two short sample notes (on RAG and on QLoRA) so the rest of the notebook has something real to retrieve from. Replace these with your own PDFs or notes any time — just re-run the indexing cells afterward.

In [6]:
os.makedirs(KNOWLEDGE_BASE_DIR, exist_ok=True)

SAMPLE_DOCS = {
    "rag_concepts.md": (
        "# Retrieval-Augmented Generation (RAG)\n\n"
        "RAG combines a retriever with a generator. Instead of relying only on what a "
        "language model memorized during training, the system first searches an external "
        "knowledge base for relevant text chunks, then feeds those chunks to the model as "
        "context alongside the user's question.\n\n"
        "## Why use RAG\n"
        "- Keeps answers grounded in real, up-to-date, or private documents\n"
        "- Reduces hallucination compared to relying purely on model memory\n"
        "- Avoids the cost of fine-tuning every time the underlying knowledge changes\n\n"
        "## Core components\n"
        "1. Document loader and chunker\n"
        "2. Embedding model that turns text into vectors\n"
        "3. Vector database for similarity search (e.g. ChromaDB, FAISS)\n"
        "4. A generator LLM that reads the retrieved context and produces the final answer\n"
    ),
    "qlora_finetuning_notes.md": (
        "# QLoRA Fine-Tuning Notes\n\n"
        "QLoRA (Quantized Low-Rank Adaptation) fine-tunes large language models cheaply by "
        "combining two ideas: loading the base model in 4-bit precision to cut memory use, "
        "and training small low-rank adapter matrices on top of the frozen base weights "
        "instead of updating every parameter.\n\n"
        "## Benefits\n"
        "- Makes fine-tuning feasible on a single consumer GPU\n"
        "- Adapter files are small (megabytes, not gigabytes) and easy to version or swap\n"
        "- Preserves most of the base model's general knowledge while specializing it\n\n"
        "## When to prefer fine-tuning over RAG\n"
        "Fine-tuning is better when you need the model to learn a *style*, *format*, or "
        "*skill* consistently. RAG is better when you need the model to know *facts* that "
        "change often or that you don't want baked into the weights.\n"
    ),
}

if not os.listdir(KNOWLEDGE_BASE_DIR):
    print("Knowledge base folder is empty — adding sample documents to get started.")
    for filename, content in SAMPLE_DOCS.items():
        with open(os.path.join(KNOWLEDGE_BASE_DIR, filename), "w", encoding="utf-8") as f:
            f.write(content)
else:
    print(f"Found {len(os.listdir(KNOWLEDGE_BASE_DIR))} existing file(s) in {KNOWLEDGE_BASE_DIR} — using those.")

Knowledge base folder is empty — adding sample documents to get started.


## 4. Load and chunk documents

Supports `.pdf`, `.md`, and `.txt` out of the box. `chunk_text` uses a simple fixed-size sliding window with overlap — easy to reason about and swap out later (e.g. for a sentence-aware or token-aware splitter) once the pipeline works end-to-end.

In [7]:
def load_text_from_file(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".pdf":
        reader = PdfReader(filepath)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif ext in (".md", ".txt"):
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    else:
        return ""

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c for c in chunks if c.strip()]

def load_and_chunk_documents(folder):
    all_chunks = []
    filepaths = glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)
    for path in filepaths:
        text = load_text_from_file(path)
        if not text.strip():
            continue
        for i, chunk in enumerate(chunk_text(text)):
            all_chunks.append({
                "id": f"{os.path.basename(path)}::{i}",
                "text": chunk,
                "source": os.path.basename(path),
            })
    return all_chunks

documents = load_and_chunk_documents(KNOWLEDGE_BASE_DIR)
print(f"Loaded {len(documents)} chunks from {KNOWLEDGE_BASE_DIR}")

Loaded 4 chunks from data/knowledge_base


## 5. Embed and index into ChromaDB

We rebuild the collection each run so re-executing the notebook doesn't duplicate chunks. For a large or frequently-updated knowledge base you'd want incremental upserts instead — worth revisiting once you're indexing more than a handful of documents.

In [8]:
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME)

def embed_text(text):
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]

print("Embedding and indexing chunks...")
for doc in tqdm(documents):
    embedding = embed_text(doc["text"])
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["text"]],
        metadatas=[{"source": doc["source"]}],
    )

print(f"Indexed {collection.count()} chunks into ChromaDB.")

Embedding and indexing chunks...


100%|██████████| 4/4 [00:01<00:00,  3.95it/s]

Indexed 4 chunks into ChromaDB.


## 6. Retrieval


In [9]:
def retrieve(query, top_k=TOP_K):
    query_embedding = embed_text(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    retrieved = []
    for text, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        retrieved.append({"text": text, "source": meta["source"], "distance": dist})
    return retrieved

## 7. Prompting and generation

The system prompt instructs Gemma 4 E2B to answer only from retrieved context and to cite sources — this is what keeps the chatbot grounded rather than free-associating.

In [10]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the provided context. "
    "If the answer isn't in the context, say you don't have enough information instead of "
    "guessing. Cite the source filename(s) you used at the end of your answer."
)

def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}" for c in retrieved_chunks
    )
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

def generate_answer(query, history=None):
    retrieved = retrieve(query)
    user_prompt = build_prompt(query, retrieved)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": user_prompt})

    response = ollama.chat(model=GENERATION_MODEL, messages=messages)
    answer = response["message"]["content"]
    return answer, retrieved

## 8. Quick sanity check

In [11]:
answer, sources = generate_answer("What is QLoRA and why is it useful?")
print(answer)
print("\nSources used:")
for s in sources:
    print(f"- {s['source']} (distance={s['distance']:.4f})")

QLoRA (Quantized Low-Rank Adaptation) fine-tunes large language models cheaply by combining two ideas: loading the base model in 4-bit precision to cut memory use, and training small low-rank adapter matrices on top of the frozen base weights instead of updating every parameter.

Benefits of QLoRA include:
*   Making fine-tuning feasible on a single consumer GPU.
*   Adapter files are small (megabytes, not gigabytes) and easy to version or swap.
*   Preserving most of the base model's general knowledge while specializing it.

Source: qlora_finetuning_notes.md

Sources used:
- qlora_finetuning_notes.md (distance=256.8313)
- rag_concepts.md (distance=405.8649)
- rag_concepts.md (distance=410.1116)
- qlora_finetuning_notes.md (distance=495.7243)


## 9. Interactive chat

Run the cell below and chat directly in the notebook's input prompt. Type `exit` or `quit` to stop. Conversation history is kept in memory for the session so follow-up questions have context.

In [12]:
def chat():
    print("RAG Chatbot ready. Type 'exit' or 'quit' to stop.\n")
    conversation_history = []
    while True:
        query = input("You: ").strip()
        if query.lower() in ("exit", "quit"):
            print("Goodbye!")
            break
        if not query:
            continue
        answer, retrieved = generate_answer(query, history=conversation_history)
        print(f"\nBot: {answer}\n")
        print("Sources:", ", ".join(sorted(set(c["source"] for c in retrieved))))
        print("-" * 60)
        conversation_history.append({"role": "user", "content": query})
        conversation_history.append({"role": "assistant", "content": answer})

# Run this cell to start chatting
chat()

RAG Chatbot ready. Type 'exit' or 'quit' to stop.


Bot: QLoRA (Quantized Low-Rank Adaptation) fine-tunes large language models cheaply by combining two ideas: loading the base model in 4-bit precision to cut memory use, and training small low-rank adapter matrices on top of the frozen base weights instead of updating every parameter.

Benefits of QLoRA include:
*   Making fine-tuning feasible on a single consumer GPU.
*   Adapter files are small (megabytes, not gigabytes) and easy to version or swap.
*   Preserving most of the base model's general knowledge while specializing it.

[qlora_finetuning_notes.md]

Sources: qlora_finetuning_notes.md, rag_concepts.md
------------------------------------------------------------

Bot: I do not have enough information to answer what model I am and what my capabilities are based on the provided context.

Sources: qlora_finetuning_notes.md, rag_concepts.md
------------------------------------------------------------
Goodbye!


## Next steps / ideas to extend this

- **Swap in your own documents** — drop PDFs, notes, or exported docs into `data/knowledge_base/`, delete `data/chroma_db/`, and re-run sections 4–5 to re-index.
- **Evaluate retrieval quality** — same spirit as the BERTScore comparison in your QLoRA notebook: build a small set of question/expected-answer pairs, run them through `generate_answer`, and score the outputs to quantify grounding quality before/after chunking or prompt changes.
- **Try a different chunk size or top-k** — these are the two levers most likely to change answer quality; worth a quick sweep once you have real documents indexed.
- **Swap the generation model** — anything pulled in Ollama works by changing `GENERATION_MODEL` (e.g. compare `gemma4:e2b` against `gemma3n:e2b` or `qwen2.5:0.5b-instruct` for a lightweight apples-to-apples comparison, tying back to your earlier fine-tuning project).
- **Package it** — once this is stable, this is a strong candidate for the flagship, end-to-end project with real evaluation metrics we discussed for your Data Science transition.